# 05 · Validation Plan — display screen, downstream format, controls

**Standard slot:** *validation plan.* **For Project 17 this means:** because de novo nanobody hit rates
are LOW, the deliverable is a **pooled display-screen plan** (designs → display → select on the TAA →
sequence winners → express → confirm), a **downstream construct** (VHH-Fc for imaging, or a CAR binder
module), a **receptor-family specificity panel**, and the mandatory **controls** (D4/D5).

This generates a structured plan file and a costed-reagent stub; it runs with no GPU.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Why a screen (not "we designed a binder")

A computational VHH design is a **hypothesis**. pLDDT is not affinity; a low pae_interaction is not
binding. De novo antibody/nanobody success rates are low, so the realistic pipeline is: **generate a
diverse pool → filter in silico → screen the pool experimentally (yeast/phage display) → recover and
characterize the winners.** Frame your designs as **screening inputs**.

## 1 · The display-screen plan `[core]`

A yeast-surface-display (or phage) screen of the filtered VHH pool against the labeled TAA. The plan
generator records the stages, the readout, and the controls so the plan is reproducible and gradable.

In [ ]:
import json, os

display_plan = {
    "format": "yeast surface display (Aga2p fusion) of the filtered VHH pool",
    "pool_source": "results/proj17_ranked.csv survivors (design_type='antibody')",
    "antigen_reagent": "recombinant TAA ectodomain (e.g., HER2 ECD), biotinylated, fluorophore-streptavidin",
    "stages": [
        "1. Synthesize the filtered VHH pool as an oligo library; clone into the display vector.",
        "2. Transform yeast; induce surface display; confirm display level (anti-tag stain).",
        "3. FACS round 1: select cells binding labeled TAA above a no-antigen gate.",
        "4. FACS rounds 2-3: increase stringency (lower antigen conc.) to enrich higher-affinity VHHs.",
        "5. Deep-sequence enriched pools; track per-design enrichment vs the input pool.",
        "6. Recover top VHHs; express solubly (see downstream format); confirm binding by SPR/BLI.",
    ],
    "readout": "FACS enrichment + NGS frequency; confirmatory SPR/BLI KD on recovered clones",
    "controls": {
        "positive": "a KNOWN anti-TAA nanobody (e.g., a published anti-HER2 VHH) spiked into the pool",
        "negative_irrelevant_antigen": "screen the same pool against an irrelevant antigen (e.g., BSA/"
                                       "a non-TAA protein) — winners must NOT enrich there",
        "negative_unrelated_binder": "an unrelated/non-binding VHH (display control, should not enrich)",
    },
    "specificity_gate": "counter-screen enriched VHHs against the receptor-family panel "
                        "(EGFR/HER3/HER4) — keep target-selective clones",
    "expectation": "LOW de novo hit rate; the screen is what turns a designed pool into real binders.",
}
os.makedirs("results", exist_ok=True)
with open("results/display_screen_plan.json", "w") as fh:
    json.dump(display_plan, fh, indent=2)
print("wrote results/display_screen_plan.json")
for s in display_plan["stages"]:
    print(" ", s)
print("\ncontrols:")
for k, v in display_plan["controls"].items():
    print(f"  {k}: {v}")

## 2 · Downstream format — imaging (VHH-Fc) or CAR binder `[core]`

A recovered, validated VHH is a *module*. Pick the translational format that matches your D0 goal:
- **VHH-Fc for imaging:** fuse the VHH to an Fc (avidity + longer half-life) or radiolabel the bare
  VHH for fast-clearing PET/SPECT tumor imaging.
- **CAR binder:** use the VHH as the antigen-binding domain of a CAR (the scFv-equivalent), linked to
  hinge/transmembrane + costimulatory + CD3ζ signaling domains.

This records the construct so the plan states what you would actually build.

In [ ]:
DOWNSTREAM = "VHH-Fc-imaging"   # or "CAR-binder"

constructs = {
    "VHH-Fc-imaging": {
        "construct": "VHH - (G4S)x linker - human IgG1 Fc (effector-silenced for imaging)",
        "purpose": "tumor-targeted imaging agent (PET/SPECT); Fc adds avidity + half-life",
        "format_notes": "bare VHH alternative for fast-clearing imaging; site-specific chelator for radiolabel",
        "assays": ["SPR/BLI KD on TAA", "cell binding on TAA+ vs TAA- lines", "(if lab) small-animal imaging"],
    },
    "CAR-binder": {
        "construct": "VHH - CD8 hinge/TM - 4-1BB - CD3zeta (VHH replaces the scFv as the binder)",
        "purpose": "CAR antigen-binding module against the TAA",
        "format_notes": "single-domain binder simplifies CAR design vs scFv; check tonic signaling",
        "assays": ["CAR-T cytotoxicity on TAA+ vs TAA- targets", "cytokine release", "specificity panel"],
    },
}
chosen = constructs[DOWNSTREAM]
print("downstream format:", DOWNSTREAM)
for k, v in chosen.items():
    print(f"  {k}: {v}")

## 3 · Controls + receptor-family specificity panel (mandatory) `[core]`

Controls are non-negotiable, even in the plan. State the **positive** (a known anti-TAA nanobody), the
**irrelevant-antigen negative** (winners must not enrich on a non-TAA protein), and the
**unrelated-binder negative** (display control). The **receptor-family specificity panel** (HER2 vs
EGFR/HER3/HER4) is the counter-screen that keeps target-selective clones.

In [ ]:
controls_and_specificity = {
    "controls": {
        "positive_control": "known anti-TAA VHH (e.g., a published anti-HER2 nanobody) — must enrich",
        "negative_irrelevant_antigen": "same pool vs BSA / a non-TAA protein — winners must NOT enrich",
        "negative_unrelated_binder": "non-binding/unrelated VHH on display — must NOT enrich",
    },
    "specificity_panel": ["HER2 (target)", "EGFR", "HER3", "HER4"],
    "specificity_decision": "keep VHHs that bind the target but not the relatives (target-selective)",
    "biparatopic_stretch": "pair a non-overlapping VHH with an overlapping one for a biparatopic/"
                           "bispecific construct (stronger avidity / dual-epitope engagement)  [stretch]",
}
import json
with open("results/controls_and_specificity.json", "w") as fh:
    json.dump(controls_and_specificity, fh, indent=2)
print(json.dumps(controls_and_specificity, indent=2))

## 4 · Costed reagent + timeline stub `[extension]`

A skeleton the student fills with real quotes/timelines. Numbers below are **EXAMPLE_DATA placeholders**
(not real quotes) — replace with vendor quotes in your D4 plan.

In [ ]:
import pandas as pd
# EXAMPLE_DATA placeholders — replace with real vendor quotes + your institution's timeline.
plan_items = pd.DataFrame([
    dict(item="VHH oligo library synthesis", purpose="display pool", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Display vector + yeast strain", purpose="surface display", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Recombinant TAA ECD (biotinylated)", purpose="FACS antigen", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="FACS sorting time", purpose="3 selection rounds", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="NGS of enriched pools", purpose="track enrichment", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Express + purify top clones", purpose="VHH-Fc / SPR", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="SPR/BLI KD on recovered clones", purpose="confirm binding", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
])
plan_items.to_csv("results/experimental_plan.csv", index=False)
print("wrote results/experimental_plan.csv (EXAMPLE_DATA placeholders — fill with real quotes)")
plan_items

## Responsible research (state in the plan)

This is a **therapeutic/diagnostic oncology** project — a designed nanobody against a human tumor
antigen for imaging or a CAR binder. In scope: diagnostic/therapeutic oncotargets. Out of scope:
anything enhancing pathogen transmissibility/virulence, toxins, or designs intended to cause harm. Any
real gene-synthesis order must go through a biosecurity-screening provider (IGSC member); wet-lab work
(including CAR-T) requires institutional biosafety/ethics approval. Do not overstate computational
designs as validated binders. See `MASTER_BLUEPRINT.md §7`.

## D4 / D5 checklist
- [ ] `results/display_screen_plan.json`: pooled yeast/phage display plan with stages + readout.
- [ ] Downstream construct chosen + recorded (VHH-Fc imaging **or** CAR binder).
- [ ] Controls (positive known nanobody, irrelevant-antigen negative, unrelated-binder negative) +
      receptor-family **specificity panel** specified.
- [ ] Costed reagent + timeline stub (EXAMPLE_DATA → real quotes).
- [ ] Responsible-research framing stated.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and Projects 14–16 reuse this antibody-family pattern (RFantibody/BoltzGen → AF2-Multimer/
IgFold → developability → `design_type="antibody"` filter → display screen).